# Accountant Agent Direct Playground

This notebook runs `AccountantAgent` directly in-process (no `/agents/{id}/chat` HTTP call).

Use it to inspect:
- token streaming behavior
- raw runtime events
- tool start/end/error calls with args and outputs
- usage events emitted by runtime

In [41]:
from __future__ import annotations

import asyncio
import importlib
import json
import subprocess
import sys
from dataclasses import asdict
from datetime import UTC, datetime
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

import httpx

ROOT = Path.cwd().resolve()
REPO_ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
AGENT_SERVICE_ROOT = REPO_ROOT / "services" / "agent"
AGENT_REQUIREMENTS = AGENT_SERVICE_ROOT / "requirements.txt"

if str(AGENT_SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(AGENT_SERVICE_ROOT))


def _parse_major_minor(ver: str) -> tuple[int, int]:
    parts = ver.split(".")
    major = int(parts[0]) if parts and parts[0].isdigit() else 0
    minor = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
    return major, minor


def _pip(args: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    cmd = [sys.executable, "-m", "pip", "--disable-pip-version-check", *args]
    return subprocess.run(cmd, capture_output=True, text=True, check=check)


def _needs_sync() -> bool:
    minimums: dict[str, tuple[int, int]] = {
        "pydantic": (2, 9),
        "pydantic-settings": (2, 6),
        "pymongo": (4, 9),
        "motor": (3, 6),
        "httpx": (0, 28),
        "langchain": (0, 3),
        "langchain-core": (0, 3),
        "langchain-openai": (0, 2),
        "langchain-anthropic": (0, 3),
        "langchain-deepseek": (0, 1),
        "langchain-ollama": (0, 2),
    }

    for package, (min_major, min_minor) in minimums.items():
        try:
            current = version(package)
        except PackageNotFoundError:
            return True
        major, minor = _parse_major_minor(current)
        if (major, minor) < (min_major, min_minor):
            return True

    # Old langchain-community versions are a common source of resolver conflicts.
    try:
        community_v = version("langchain-community")
        c_major, c_minor = _parse_major_minor(community_v)
        if (c_major, c_minor) < (0, 3):
            return True
    except PackageNotFoundError:
        pass

    # Ensure optional extras expected by runtime model imports exist.
    for mod_name in ("email_validator", "bson"):
        try:
            importlib.import_module(mod_name)
        except ImportError:
            return True

    return False


def ensure_agent_runtime_dependencies() -> bool:
    """Return True when a kernel restart is required."""
    if not AGENT_REQUIREMENTS.exists():
        raise FileNotFoundError(f"Requirements file not found: {AGENT_REQUIREMENTS}")

    if not _needs_sync():
        print("Notebook kernel already aligned with agent requirements.")
        return False

    print("Aligning notebook kernel with agent requirements...")

    # Remove known conflicting package line before sync.
    try:
        community_v = version("langchain-community")
        c_major, c_minor = _parse_major_minor(community_v)
        if (c_major, c_minor) < (0, 3):
            print("Removing conflicting package: langchain-community")
            _pip(["uninstall", "-y", "langchain-community"], check=False)
    except PackageNotFoundError:
        pass

    install_result = _pip(
        [
            "install",
            "--upgrade",
            "-r",
            str(AGENT_REQUIREMENTS),
        ],
        check=True,
    )

    # Keep output concise but still informative.
    if install_result.stdout.strip():
        print("pip install completed.")
    if install_result.stderr.strip():
        print("pip reported warnings; restart is still required.")

    print("\nDependencies were updated in this kernel environment.")
    print("Restart the Jupyter kernel, then run this notebook from the first cell.")
    return True


# Keep local notebook environment aligned with agent service runtime deps.
RESTART_REQUIRED = ensure_agent_runtime_dependencies()

# Stable defaults available even when setup is paused for kernel restart.
X_USER_ID = globals().get("X_USER_ID", "69ee3344826015e6593e6c9e")
AGENT_ID = globals().get("AGENT_ID", "69ee334437c7ac857d865c8d")
STREAM_TOKENS = bool(globals().get("STREAM_TOKENS", True))
MAX_PREVIEW_CHARS = int(globals().get("MAX_PREVIEW_CHARS", 500))

if RESTART_REQUIRED:
    print("\nSetup paused until kernel restart.")
    print("After restart, rerun this cell and continue with the notebook.")
else:
    from app.clients.business import BusinessClient
    from app.config import settings
    from app.models.shared.conversation import MessageSchema
    from app.repositories.agent_repo import AgentRepository
    from app.runtime.accountant import AccountantAgent
    from app.runtime.hooks import AgentRunEvent
    from app.runtime.providers.factory import LLMProviderFactory
    from app.runtime.tool_context import ToolContext
    from app.services.companybook_service import CompanyBookService
    from app.utils.db import close_db, connect_db, get_database

    # Notebook runs on host, so Docker-internal hostnames like "mongodb" are not resolvable.
    # Prefer localhost port mappings from docker-compose.dev.yml unless user already set explicit URLs.
    if "localhost" not in settings.mongodb_url and "127.0.0.1" not in settings.mongodb_url:
        settings.mongodb_url = "mongodb://localhost:27017"
    if "localhost" not in settings.business_service_url and "127.0.0.1" not in settings.business_service_url:
        settings.business_service_url = "http://localhost:8005"
    if "localhost" not in settings.knowledge_service_url and "127.0.0.1" not in settings.knowledge_service_url:
        settings.knowledge_service_url = "http://localhost:8003"

    print(f"Repo root: {REPO_ROOT}")
    print(f"Agent service root: {AGENT_SERVICE_ROOT}")
    print(f"Mongo URL: {settings.mongodb_url}")
    print(f"Business URL: {settings.business_service_url}")
    print(f"Knowledge URL: {settings.knowledge_service_url}")

Notebook kernel already aligned with agent requirements.
Repo root: /Users/valentinbarakov/proecti/agents
Agent service root: /Users/valentinbarakov/proecti/agents/services/agent
Mongo URL: mongodb://localhost:27017
Business URL: http://localhost:8005
Knowledge URL: http://localhost:8003


In [42]:
if "AccountantAgent" not in globals() or "AgentRunEvent" not in globals():
    raise RuntimeError(
        "Setup imports are not loaded. Run cell 1 first. "
        "If it says restart is required, restart kernel and rerun from cell 1."
    )


def _preview(value: Any, limit: int | None = None) -> str:
    if limit is None:
        limit = int(globals().get("MAX_PREVIEW_CHARS", 500))

    if value is None:
        text = ""
    elif isinstance(value, str):
        text = value
    else:
        try:
            text = json.dumps(value, ensure_ascii=True, default=str)
        except Exception:
            text = repr(value)
    return text if len(text) <= limit else text[:limit] + " ..."


class TraceAccountantAgent(AccountantAgent):
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.trace_events: list[dict[str, Any]] = []

    async def on_event(self, event: AgentRunEvent) -> AgentRunEvent | None:
        raw = event.raw
        data = raw.get("data") if isinstance(raw.get("data"), dict) else {}
        self.trace_events.append(
            {
                "event": raw.get("event"),
                "name": raw.get("name"),
                "run_id": raw.get("run_id"),
                "input": data.get("input"),
                "output": data.get("output"),
                "error": data.get("error"),
            }
        )
        return event


def extract_tool_timeline(trace_events: list[dict[str, Any]]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for idx, event in enumerate(trace_events, start=1):
        event_name = str(event.get("event") or "")
        if event_name not in {"on_tool_start", "on_tool_end", "on_tool_error"}:
            continue

        status = "start" if event_name == "on_tool_start" else "end" if event_name == "on_tool_end" else "error"
        rows.append(
            {
                "order": idx,
                "status": status,
                "tool": str(event.get("name") or "unknown"),
                "run_id": str(event.get("run_id") or "n/a"),
                "input": event.get("input"),
                "output": event.get("output"),
                "error": event.get("error"),
            }
        )
    return rows


def print_tool_timeline(trace_events: list[dict[str, Any]]) -> None:
    rows = extract_tool_timeline(trace_events)
    print("\nTool timeline")
    print("=" * 110)
    if not rows:
        print("No tool events captured in runtime event stream.")
        return

    counts: dict[str, int] = {}
    for row in rows:
        name = row["tool"]
        counts[name] = counts.get(name, 0) + 1

    print("Tool counts:")
    for name, count in sorted(counts.items(), key=lambda x: (-x[1], x[0])):
        print(f"- {name}: {count}")

    print("\nCalls:")
    for row in rows:
        payload = row["input"] if row["status"] == "start" else row["output"] if row["status"] == "end" else row["error"]
        print(
            f"{row['order']:02d}. [{row['status']:5}] {row['tool']} "
            f"| run={row['run_id']} | {_preview(payload)}"
        )

In [43]:
if "MessageSchema" not in globals() or "AgentRepository" not in globals() or "LLMProviderFactory" not in globals():
    raise RuntimeError(
        "Runtime dependencies are not loaded. Run cell 1 first. "
        "If it performed installs, restart kernel and rerun from cell 1."
    )


async def run_direct_accountant_turn(
    *,
    user_id: str,
    agent_id: str,
    message: str,
    history: list[MessageSchema] | None = None,
    stream_tokens: bool | None = None,
) -> dict[str, Any]:
    history = history or []
    if stream_tokens is None:
        stream_tokens = bool(globals().get("STREAM_TOKENS", True))

    try:
        await connect_db()
    except Exception as exc:
        raise RuntimeError(
            "Failed to connect to MongoDB from notebook. "
            "Ensure docker services are running and mongo is reachable at settings.mongodb_url. "
            f"Current mongodb_url={settings.mongodb_url!r}"
        ) from exc

    business_http = httpx.AsyncClient(
        base_url=settings.business_service_url,
        timeout=settings.business_timeout_seconds,
    )
    knowledge_http = httpx.AsyncClient(
        base_url=settings.knowledge_service_url,
        timeout=settings.knowledge_timeout_seconds,
    )

    try:
        db = get_database()
        agent_repo = AgentRepository(db)
        agent = await agent_repo.get_by_id(user_id, agent_id)
        if agent is None:
            raise ValueError(f"Agent not found: {agent_id}")
        if agent.agent_type != "accountant":
            raise ValueError(f"Expected accountant agent, got: {agent.agent_type}")

        provider = LLMProviderFactory.create(agent.config.provider)
        llm = provider.create_chat_model(agent.config.model, agent.config.temperature)

        tool_context = ToolContext(
            business_client=BusinessClient(business_http),
            companybook_service=CompanyBookService(
                api_key=settings.companybook_api_key,
                base_url=settings.companybook_base_url,
                timeout=settings.companybook_timeout_seconds,
            ),
            knowledge_http=knowledge_http,
        )

        runtime = TraceAccountantAgent(
            agent=agent,
            llm=llm,
            user_id=user_id,
            tool_context=tool_context,
        )

        chunks: list[str] = []
        async for token in runtime.run(message, history):
            chunks.append(token)
            if stream_tokens:
                print(token, end="", flush=True)
        if stream_tokens:
            print()

        usage_events = [asdict(event) for event in runtime.consume_usage_events()]
        assistant_text = "".join(chunks)

        return {
            "assistant_text": assistant_text,
            "trace_events": runtime.trace_events,
            "tool_timeline": extract_tool_timeline(runtime.trace_events),
            "usage_events": usage_events,
        }
    finally:
        await business_http.aclose()
        await knowledge_http.aclose()
        close_db()


def run_turn_sync(
    *,
    user_id: str,
    agent_id: str,
    message: str,
    history: list[MessageSchema] | None = None,
    stream_tokens: bool | None = None,
    require_preflight: bool = True,
) -> dict[str, Any]:
    if require_preflight and "preflight_check_sync" in globals():
        report = preflight_check_sync(raise_on_fail=False)
        failed = [name for name, row in report.items() if not bool(row.get("ok"))]
        if failed:
            details = "\n".join(
                f"- {name}: {report[name].get('detail')}" for name in failed
            )
            raise RuntimeError(
                "Preflight failed before agent run.\n"
                f"{details}\n"
                "Fix connectivity first, then rerun."
            )

    coro = run_direct_accountant_turn(
        user_id=user_id,
        agent_id=agent_id,
        message=message,
        history=history,
        stream_tokens=stream_tokens,
    )

    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        # Jupyter runs an active event loop; patch it for nested execution.
        try:
            import nest_asyncio  # type: ignore
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "nest-asyncio>=1.6.0"])
            import nest_asyncio  # type: ignore
        nest_asyncio.apply()
        return loop.run_until_complete(coro)

    return asyncio.run(coro)

In [44]:
def preflight_check_sync(*, timeout_seconds: float = 3.0, raise_on_fail: bool = False) -> dict[str, dict[str, Any]]:
    """Validate host connectivity for dependencies used by direct runtime runs."""
    report: dict[str, dict[str, Any]] = {
        "mongodb": {"ok": False, "detail": ""},
        "business_health": {"ok": False, "detail": ""},
        "business_data": {"ok": False, "detail": ""},
        "knowledge": {"ok": False, "detail": ""},
    }

    # MongoDB ping
    try:
        from pymongo import MongoClient

        client = MongoClient(settings.mongodb_url, serverSelectionTimeoutMS=int(timeout_seconds * 1000))
        client.admin.command("ping")
        client.close()
        report["mongodb"] = {"ok": True, "detail": f"reachable at {settings.mongodb_url}"}
    except Exception as exc:
        report["mongodb"] = {"ok": False, "detail": f"{settings.mongodb_url} -> {exc}"}

    headers = {"x-user-id": str(globals().get("X_USER_ID", ""))}

    # Business /health (liveness only)
    business_health = f"{settings.business_service_url.rstrip('/')}/health"
    try:
        r = httpx.get(business_health, timeout=timeout_seconds)
        report["business_health"] = {
            "ok": r.status_code == 200,
            "detail": f"{business_health} -> {r.status_code}",
        }
    except Exception as exc:
        report["business_health"] = {"ok": False, "detail": f"{business_health} -> {exc}"}

    # Business data endpoint probe (DB-backed route used by tools)
    business_data_url = f"{settings.business_service_url.rstrip('/')}/companies"
    try:
        r = httpx.get(
            business_data_url,
            headers=headers,
            params={"limit": 1, "offset": 0},
            timeout=max(timeout_seconds, 8.0),
        )
        ok = r.status_code == 200
        detail = f"{business_data_url} -> {r.status_code}"
        if ok:
            try:
                payload = r.json()
                count = len(payload) if isinstance(payload, list) else -1
                detail += f" (items={count})"
            except Exception:
                pass
        else:
            detail += f" body={r.text[:180]}"
        report["business_data"] = {"ok": ok, "detail": detail}
    except Exception as exc:
        report["business_data"] = {"ok": False, "detail": f"{business_data_url} -> {exc}"}

    # Knowledge /health
    knowledge_health = f"{settings.knowledge_service_url.rstrip('/')}/health"
    try:
        r = httpx.get(knowledge_health, timeout=timeout_seconds)
        report["knowledge"] = {
            "ok": r.status_code == 200,
            "detail": f"{knowledge_health} -> {r.status_code}",
        }
    except Exception as exc:
        report["knowledge"] = {"ok": False, "detail": f"{knowledge_health} -> {exc}"}

    print("Preflight check")
    print("=" * 110)
    for name in ("mongodb", "business_health", "business_data", "knowledge"):
        status = "OK" if report[name]["ok"] else "FAIL"
        print(f"- {name:14} [{status}] {report[name]['detail']}")

    failed = [name for name, row in report.items() if not bool(row.get("ok"))]
    if failed and raise_on_fail:
        details = "\n".join(f"- {name}: {report[name]['detail']}" for name in failed)
        raise RuntimeError(f"Preflight failed:\n{details}")

    return report


def append_turn_to_history(history: list[MessageSchema], user_message: str, assistant_message: str) -> list[MessageSchema]:
    now = datetime.now(UTC)
    history.append(MessageSchema(role="user", content=user_message, created_at=now))
    history.append(MessageSchema(role="assistant", content=assistant_message, created_at=now))
    return history


def print_usage_events(usage_events: list[dict[str, Any]]) -> None:
    print("\nLLM usage events")
    print("=" * 110)
    if not usage_events:
        print("No usage events captured.")
        return

    for idx, event in enumerate(usage_events, start=1):
        print(
            f"{idx:02d}. model={event.get('model')} provider={event.get('provider')} "
            f"run_id={event.get('run_id')} duration_ms={event.get('duration_ms')} "
            f"tokens(in={event.get('input_tokens')} out={event.get('output_tokens')} total={event.get('total_tokens')})"
        )

In [45]:
# Interactive playground cell
SESSION_HISTORY: list[MessageSchema] = []

_ = preflight_check_sync()

USER_MESSAGE = "Get my latest invoice from November 2025."

result = run_turn_sync(
    user_id=X_USER_ID,
    agent_id=AGENT_ID,
    message=USER_MESSAGE,
    history=SESSION_HISTORY,
    stream_tokens=True,
    require_preflight=True,
)

print("\nAssistant response")
print("=" * 110)
print(result["assistant_text"])

print_tool_timeline(result["trace_events"])
print_usage_events(result["usage_events"])

append_turn_to_history(SESSION_HISTORY, USER_MESSAGE, result["assistant_text"])
print(f"\nSession history messages: {len(SESSION_HISTORY)}")

Preflight check
- mongodb        [OK] reachable at mongodb://localhost:27017
- business_health [OK] http://localhost:8005/health -> 200
- business_data  [OK] http://localhost:8005/companies -> 200 (items=1)
- knowledge      [OK] http://localhost:8003/health -> 200
Preflight check
- mongodb        [OK] reachable at mongodb://localhost:27017
- business_health [OK] http://localhost:8005/health -> 200
- business_data  [OK] http://localhost:8005/companies -> 200 (items=1)
- knowledge      [OK] http://localhost:8003/health -> 200
Your latest invoice from November 2025 is invoice number INV-2025-00003 issued on 24th November 2025. It was issued to "РОБО ЛАБ" with a total amount of 1211.12 EUR (including 201.85 EUR VAT). The invoice is for payroll processing services and has been paid. If you need more details or a copy of this invoice, please let me know.

Assistant response
Your latest invoice from November 2025 is invoice number INV-2025-00003 issued on 24th November 2025. It was issued to 

In [46]:
# Optional: repeatable tool-heavy cases
_ = preflight_check_sync()

CASES = [
    "Create an invoice draft to Contoso for 1500 EUR software services.",
    "Record expense: office supplies 320 BGN paid today.",
    "Use rag_search once and explain VAT registration threshold in Bulgaria.",
]

for i, prompt in enumerate(CASES, start=1):
    print(f"\n--- Case {i}: {prompt}")
    out = run_turn_sync(
        user_id=X_USER_ID,
        agent_id=AGENT_ID,
        message=prompt,
        history=SESSION_HISTORY,
        stream_tokens=False,
        require_preflight=True,
    )
    print(f"Assistant chars: {len(out['assistant_text'])}")
    print_tool_timeline(out["trace_events"])
    print_usage_events(out["usage_events"])
    append_turn_to_history(SESSION_HISTORY, prompt, out["assistant_text"])

print(f"\nFinal session history messages: {len(SESSION_HISTORY)}")

Preflight check
- mongodb        [OK] reachable at mongodb://localhost:27017
- business_health [OK] http://localhost:8005/health -> 200
- business_data  [OK] http://localhost:8005/companies -> 200 (items=1)
- knowledge      [OK] http://localhost:8003/health -> 200

--- Case 1: Create an invoice draft to Contoso for 1500 EUR software services.
Preflight check
- mongodb        [OK] reachable at mongodb://localhost:27017
- business_health [OK] http://localhost:8005/health -> 200
- business_data  [OK] http://localhost:8005/companies -> 200 (items=1)
- knowledge      [OK] http://localhost:8003/health -> 200
Assistant chars: 378

Tool timeline
Tool counts:
- resolve_partner_by_name: 2

Calls:
35. [start] resolve_partner_by_name | run=019df82a-8392-7382-ae1c-e13d093d497f | {"name": "Contoso", "kind": "client"}
36. [end  ] resolve_partner_by_name | run=019df82a-8392-7382-ae1c-e13d093d497f | No partner matching 'Contoso' was found. Ask the user for partner details or create the partner first.



In [48]:
# LangChain graph printer (actual runtime chain)
# This builds the same tool-calling runnable as BaseAgent.run and prints its graph.

if "MessageSchema" not in globals() or "AgentRepository" not in globals() or "LLMProviderFactory" not in globals():
    raise RuntimeError(
        "Runtime dependencies are not loaded. Run cell 1 first. "
        "If it performed installs, restart kernel and rerun from cell 1."
    )

from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


async def print_accountant_runtime_graph(*, user_id: str, agent_id: str) -> None:
    try:
        await connect_db()
    except Exception as exc:
        raise RuntimeError(
            "Failed to connect to MongoDB for graph build. "
            f"Current mongodb_url={settings.mongodb_url!r}"
        ) from exc

    business_http = httpx.AsyncClient(
        base_url=settings.business_service_url,
        timeout=settings.business_timeout_seconds,
    )
    knowledge_http = httpx.AsyncClient(
        base_url=settings.knowledge_service_url,
        timeout=settings.knowledge_timeout_seconds,
    )

    try:
        db = get_database()
        agent_repo = AgentRepository(db)
        agent = await agent_repo.get_by_id(user_id, agent_id)
        if agent is None:
            raise ValueError(f"Agent not found: {agent_id}")
        if agent.agent_type != "accountant":
            raise ValueError(f"Expected accountant agent, got: {agent.agent_type}")

        provider = LLMProviderFactory.create(agent.config.provider)
        llm = provider.create_chat_model(agent.config.model, agent.config.temperature)

        tool_context = ToolContext(
            business_client=BusinessClient(business_http),
            companybook_service=CompanyBookService(
                api_key=settings.companybook_api_key,
                base_url=settings.companybook_base_url,
                timeout=settings.companybook_timeout_seconds,
            ),
            knowledge_http=knowledge_http,
        )

        runtime = TraceAccountantAgent(
            agent=agent,
            llm=llm,
            user_id=user_id,
            tool_context=tool_context,
        )

        tools = runtime.get_tools()
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", runtime.get_system_prompt()),
                MessagesPlaceholder("chat_history"),
                ("human", "{input}"),
                MessagesPlaceholder("agent_scratchpad"),
            ]
        )

        runnable_agent = create_tool_calling_agent(llm, tools, prompt)
        graph = runnable_agent.get_graph()

        print("Runnable graph (ASCII):")
        print("=" * 110)
        try:
            graph.print_ascii()
        except Exception as exc:
            if "grandalf" in str(exc).lower():
                print("Installing missing graph dependency: grandalf")
                subprocess.check_call([sys.executable, "-m", "pip", "install", "grandalf>=0.8"])
                graph.print_ascii()
            else:
                print(f"print_ascii not available: {exc}")

        print("\nRunnable graph (Mermaid):")
        print("=" * 110)
        try:
            mermaid = graph.draw_mermaid()
            print(mermaid)
        except Exception as exc:
            print(f"draw_mermaid not available: {exc}")

        # Also show executor wrapper nodes if available.
        executor = AgentExecutor(agent=runnable_agent, tools=tools, max_iterations=settings.max_agent_iterations)
        print("\nExecutor graph (ASCII):")
        print("=" * 110)
        try:
            executor.get_graph().print_ascii()
        except Exception as exc:
            if "grandalf" in str(exc).lower():
                print("Installing missing graph dependency: grandalf")
                subprocess.check_call([sys.executable, "-m", "pip", "install", "grandalf>=0.8"])
                executor.get_graph().print_ascii()
            else:
                print(f"executor graph print unavailable: {exc}")

    finally:
        await business_http.aclose()
        await knowledge_http.aclose()
        close_db()


def print_accountant_runtime_graph_sync(*, user_id: str, agent_id: str) -> None:
    coro = print_accountant_runtime_graph(user_id=user_id, agent_id=agent_id)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        try:
            import nest_asyncio  # type: ignore
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "nest-asyncio>=1.6.0"])
            import nest_asyncio  # type: ignore
        nest_asyncio.apply()
        loop.run_until_complete(coro)
        return

    asyncio.run(coro)


# Usage:
print_accountant_runtime_graph_sync(user_id=X_USER_ID, agent_id=AGENT_ID)

Runnable graph (ASCII):
Installing missing graph dependency: grandalf



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


ImportError: Install grandalf to draw graphs: `pip install grandalf`.